In [ ]:
!pip install requests pandas beautifulsoup4

In [ ]:
# -*- coding: utf-8 -*-
"""
네이버 뉴스 Open API + BeautifulSoup 혼합 수집 파이프라인
- 키워드: "캄보디아"
- 최근 7일 기사 50건 이상 수집
- 수집 항목: 제목, 날짜, 언론사, 링크, 원문 링크
- Pandas 정리 → CSV 저장
- 분석: 일자별 집계, 언론사 TOP10, 제목 단어 TOP20 (불용어 제외)
"""

import os
import time
import re
import html
from collections import Counter
from urllib.parse import urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup
from email.utils import parsedate_to_datetime
from datetime import datetime, timedelta, timezone

# =========================
# 환경 설정
# =========================
NAVER_CLIENT_ID = '발급받은 Client ID'
NAVER_CLIENT_SECRET = '발급받은 Client Secret'

KEYWORD = "캄보디아"
DAYS = 7
TARGET_COUNT = 60             # 최소 50건 이상 확보를 위해 여유 있게
DISPLAY = 100                 # 한 번에 가져올 수 있는 최대(네이버 뉴스 API 기준)
MAX_START = 1000              # API start 인덱스 상한
SLEEP = 0.3                   # API/스크래핑 예의상 딜레이(초)
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0.0.0 Safari/537.36"
)

HEADERS_API = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET,
}
HEADERS_WEB = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}

KST = timezone(timedelta(hours=9))
now_kst = datetime.now(KST)
threshold_dt = now_kst - timedelta(days=DAYS)

session = requests.Session()
session.headers.update(HEADERS_WEB)


# =========================
# 유틸 함수
# =========================
"""1) 네이버 API가 <b>강조</b> 형태로 내려주는 태그 및 HTML 엔티티 제거"""
def clean_html_tags(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"<.*?>", " ", text)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

"""2) 링크가 news.naver.com 도메인인지 확인"""
def is_naver_news(url: str) -> bool:
    try:
        netloc = urlparse(url).netloc.lower()
        return "news.naver.com" in netloc
    except Exception:
        return False

"""
3)  원문 도메인으로 언론사 명 추정 (정확X)
ex) www.khan.co.kr -> khan
"""
def domain_to_press(domain: str) -> str:
    if not domain:
        return ""
    domain = domain.lower()
    domain = re.sub(r"^(www|m|news|media)\.", "", domain)  # 흔한 서브도메인 제거
    base = domain.split(".")[0]
    return base.capitalize()


"""
4)  BeautifulSoup으로 네이버 뉴스 상세 페이지에서 언론사 이름 추출
    - 메타 태그 -> 로고 alt -> 구조적 데이터 순차 시도
"""
def extract_press_from_naver_article(url: str) -> str:
    try:
        resp = session.get(url, timeout=10)
        if resp.status_code != 200:
            return ""
        soup = BeautifulSoup(resp.text, "html.parser")

        # 1) 대표 메타 태그들
        for sel in [
            'meta[property="og:article:author"]',
            'meta[name="twitter:creator"]',
            'meta[property="og:site_name"]',
            'meta[name="cpname"]',
        ]:
            tag = soup.select_one(sel)
            if tag and tag.get("content"):
                candidate = tag.get("content").strip()
                if candidate:
                    return candidate

        # 2) 로고 이미지 alt 속성
        tag = soup.select_one('.media_end_head_top_logo img[alt], .press_logo img[alt]')
        if tag and tag.get("alt"):
            return tag.get("alt").strip()

        # 3) 기사 상단 영역 내 텍스트
        for sel in [
            ".media_end_head_top_logo",  # 새 디자인
            ".press_logo",               # 구 디자인
        ]:
            el = soup.select_one(sel)
            if el:
                txt = el.get_text(strip=True)
                if txt:
                    return txt

    except Exception:
        return ""
    return ""

"""
5)  네이버 뉴스 Open API에서 검색 결과 가져오기 (페이지 단위)
    - 날짜 필터는 API 미지원, 후처리로 필터링
"""
def fetch_naver_news_items(keyword: str, display: int = DISPLAY):
    all_items = []
    for start in range(1, MAX_START + 1, display):
        params = {
            "query": keyword,
            "display": display,
            "start": start,
            "sort": "date",   # 최신순
        }
        resp = requests.get(
            "https://openapi.naver.com/v1/search/news.json",
            params=params,
            headers=HEADERS_API,
            timeout=10,
        )
        if resp.status_code != 200:
            print(f"[경고] API 호출 실패 start={start} status={resp.status_code} body={resp.text[:200]}")
            break

        data = resp.json()
        items = data.get("items", [])
        if not items:
            break

        all_items.extend(items)
        # API 남용 방지
        time.sleep(SLEEP)

    return all_items


# =========================
# 수집 실행
# =========================
print(f"[정보] 키워드='{KEYWORD}', 최근 {DAYS}일, 최소 {TARGET_COUNT}건 수집 시작...")

raw_items = fetch_naver_news_items(KEYWORD, display=DISPLAY)
print(f"[정보] API 총 수신 아이템: {len(raw_items)} (날짜 후필터 전)")

records = []
seen = set()  # 중복 제거 (제목+링크 기준)

for it in raw_items:
    title_raw = it.get("title", "")
    originallink = it.get("originallink") or ""
    link = it.get("link") or ""
    pub_raw = it.get("pubDate", "")

    # 날짜 파싱 (RFC-822) + 타임존 유지 → KST로 변환
    try:
        pub_dt = parsedate_to_datetime(pub_raw)
        if pub_dt.tzinfo is None:
            pub_dt = pub_dt.replace(tzinfo=timezone.utc)
        pub_dt_kst = pub_dt.astimezone(KST)
    except Exception:
        # 날짜 파싱 실패 시 스킵
        continue

    # 최근 N일 필터
    if pub_dt_kst < threshold_dt:
        continue

    # 텍스트 정리
    title = clean_html_tags(title_raw)

    # 중복 스킵 (같은 제목+링크)
    dup_key = (title.lower(), link)
    if dup_key in seen:
        continue
    seen.add(dup_key)

    # 언론사 추출: 네이버 뉴스면 파싱, 아니면 originallink 도메인으로 추측
    press = ""
    if is_naver_news(link):
        press = extract_press_from_naver_article(link)
        time.sleep(SLEEP)  # 과한 요청 방지
    if not press:
        domain = urlparse(originallink).netloc if originallink else urlparse(link).netloc
        press = domain_to_press(domain)

    records.append({
        "title": title,
        "pub_datetime_kst": pub_dt_kst.isoformat(timespec="seconds"),
        "pub_date": pub_dt_kst.date().isoformat(),
        "press": press,
        "link": link,
        "originallink": originallink,
        "keyword": KEYWORD,
    })

    # 목표 개수 도달 시 종료
    if len(records) >= TARGET_COUNT:
        break

print(f"[정보] 기간 필터 및 정리 후 수집 건수: {len(records)}")

if len(records) < 50:
    print("[주의] 최근 7일 내 기사 50건을 확보하지 못했습니다. 키워드/기간/TARGET_COUNT를 조정하세요.")

# =========================
# DataFrame & 저장
# =========================
df = pd.DataFrame(records)
if not df.empty:
    # 컬럼 순서 정리
    df = df[["pub_date", "pub_datetime_kst", "press", "title", "link", "originallink", "keyword"]]

save_name = f"naver_news_{KEYWORD}_{now_kst.strftime('%Y%m%d')}.csv"
df.to_csv(save_name, index=False, encoding="utf-8-sig")
print(f"[완료] CSV 저장: {save_name}, shape={df.shape}")


[정보] 키워드='캄보디아', 최근 7일, 최소 60건 수집 시작...
[정보] API 총 수신 아이템: 1000 (날짜 후필터 전)
[정보] 기간 필터 및 정리 후 수집 건수: 60
[완료] CSV 저장: naver_news_캄보디아_20251019.csv, shape=(60, 7)


In [3]:
# =========================
# 분석
# =========================

if not df.empty:
    # 1) 일자별 기사 수
    by_date = df.groupby("pub_date").size().sort_index(ascending=True)
    print("\n[분석] 일자별 기사 수")
    print(by_date)

    # 2) 언론사별 기사 수 TOP 10
    top_press = df["press"].value_counts().head(10)
    print("\n[분석] 언론사별 기사 수 TOP 10")
    print(top_press)

    # 3) 제목에서 가장 많이 등장한 단어 TOP 20 (불용어 제외)
    titles_joined = " ".join(df["title"].astype(str))
    tokens = re.findall(r"[가-힣A-Za-z]{2,}", titles_joined)

    stopwords = {
        "및", "등", "더", "또", "이후", "관련", "대한", "에서", "으로", "에게", "이고", "한다",
        "했다", "하기", "한다는", "제", "뉴스", "속보", "영상", "단독", "종합", "오늘", "내일",
        "이어서", "따라", "기자", "사진", "일보", "신문", "경제", "사회", "정치", "국제", "전문"
        "캄보디아", "캄보디아서", "캄보디아에", "캄보디아의", "캄보디아서도",
        "news", "the", "this",
    }

    norm_tokens = []
    for w in tokens:
        if re.match(r"[A-Za-z]+$", w):
            w = w.lower()
        if len(w) < 2:
            continue
        if w in stopwords:
            continue
        norm_tokens.append(w)

    freq = Counter(norm_tokens)
    top20 = freq.most_common(20)

    print("\n[분석] 제목 단어 TOP 20 (불용어 제외)")
    for word, cnt in top20:
        print(f"{word}\t{cnt}")
else:
    print("[정보] 수집된 데이터가 없어 분석을 건너뜁니다.")



[분석] 일자별 기사 수
pub_date
2025-10-19    60
dtype: int64

[분석] 언론사별 기사 수 TOP 10
press
MBN | 네이버       7
경향신문 | 네이버      5
MBC | 네이버       4
SBS | 네이버       2
Obsnews         2
문화일보 | 네이버      2
Ksmnews         2
파이낸셜뉴스 | 네이버    2
Kjdaily         2
연합뉴스 | 네이버      2
Name: count, dtype: int64

[분석] 제목 단어 TOP 20 (불용어 제외)
캄보디아	37
경찰	9
송환	8
귀국	6
한국인	4
대학생	4
피의자	4
구속영장	4
국내	4
수사	4
기후특사단	4
신청	4
사망	3
부검	3
한국	3
송환자	3
검토	3
정부	3
광주	3
대사관	3
